# 設備見積もり/制御システム — ピッチCGレンダリング (Colab)

**ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択。あとは **ランタイム → すべて実行** でOK（ワンクリック）。

- エンジン: `documentary_gpu`（FLUX実写 → 2.5Dパララックス → グレード/テキスト）
- **A**: ベクター図解層 `overlay_vector`（線/ノード/データ流をアニメで実写の上に）
- **C**: Manimヒーロー図 `manim_scenes.AdapterSpine`（背骨←アダプタ→設備）
- 絵コンテ: `scenes/setsubi_pitch.json`（9カット・約72秒）

**事前に1回だけ**: 🔑Secretsに `HF_TOKEN` を登録（FLUX高画質化／未登録でもSDXLで動く）。
依存導入は初回のみ数分。通しで焼くだけならセル5・6（試写）は飛ばして可。

In [ ]:
# ═══ セル1: コード取り込み + Drive マウント ═══
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/documentary_gpu'
REPO = 'https://github.com/task2000jp/documentary_gpu.git'  # public・無認証clone可
if os.path.isdir(WORK):
    subprocess.run(['git', '-C', WORK, 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', REPO, WORK], check=True)
print('WORK =', WORK)

In [ ]:
# ═══ セル2: 依存インストール（パララックス + FLUX画像生成）═══
# torch/CUDA は Colab プリインストール版を使う（上書きしない）。
%cd $WORK
!apt-get -qq install -y ffmpeg fonts-ipafont fonts-noto-cjk > /dev/null
!pip install -q transformers Pillow numpy imageio imageio-ffmpeg soundfile edge-tts
# FLUX (sd_generated 背景) 用。bitsandbytes=NF4量子化（FLUX ~12GB→~7GB）
!pip install -q "diffusers>=0.31" accelerate sentencepiece bitsandbytes

In [ ]:
# ═══ セル3: Manim 依存（ヒーロー図カット用）═══
# AdapterSpine カットを使うときだけ実行。cairo/pango が必要。
%cd $WORK
!apt-get -qq install -y libcairo2-dev libpango1.0-dev pkg-config > /dev/null
!pip install -q manim
# 日本語フォント(Noto CJK)はセル2で導入済み。manim の Text(font='Noto Sans CJK JP') で使用。

In [ ]:
# ═══ セル3b: HuggingFace 認証（FLUX用・推奨）═══
# 🔑Colab左の鍵アイコン → Secrets に HF_TOKEN を登録（ノートブックアクセスON）。
# 未登録ならスキップ→SDXLフォールバック（FLUXより低画質・低速だが動く）。
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(os.environ['HF_TOKEN'])
    print('HF認証OK → FLUX有効')
except Exception as e:
    print('HF_TOKEN未設定 → SDXLフォールバックで描画:', e)

In [ ]:
# ═══ セル4b: 画像キャッシュ強制クリア（任意・初回や絵を作り直す時だけ）═══
# 旧SDXL画像が残っていると直しても再生成されない。改善版の絵にしたい時に1回実行。
# 普段は不要（キャッシュ再利用で高速）。
%cd $WORK
!rm -f build/img_cache/sp_*.png && echo 'sp_* キャッシュ削除 → 次回再生成'

In [ ]:
# ═══ セル4: 環境チェック ═══
%cd $WORK
!python scripts/pipeline.py doctor
import shutil
print('manim:', '✅' if shutil.which('manim') else '⚪ 未導入(セル3)')

In [ ]:
# ═══ セル5: 試写A — FLUX実写 + ベクター図解（アダプタのカット）═══
%cd $WORK
!python scripts/pipeline.py render-scene scenes/setsubi_pitch.json sp_adapter
from IPython.display import Video
Video('build/clips/sp_adapter.mp4', embed=True, width=720)

In [ ]:
# ═══ セル6: 試写C — Manimヒーロー図（背骨←アダプタ→設備）═══
%cd $WORK
!python scripts/pipeline.py render-scene scenes/setsubi_pitch.json sp_spine_hero
from IPython.display import Video
Video('build/clips/sp_spine_hero.mp4', embed=True, width=720)

In [ ]:
# ═══ セル7: 本番 — 全9カットをレンダリング（約72秒の動画）═══
%cd $WORK
!python scripts/pipeline.py render setsubi_pitch
from IPython.display import Video
Video('build/chapters/setsubi_pitch.mp4', embed=True, width=720)

In [ ]:
# ═══ セル8（任意）: BGMを重ねる ═══
# Drive の MyDrive/setsubi_bgm.mp3 を敷く場合のみ。無ければスキップ可。
%cd $WORK
import os
BGM = '/content/drive/MyDrive/setsubi_bgm.mp3'
if os.path.exists(BGM):
    !ffmpeg -y -i build/chapters/setsubi_pitch.mp4 -i "$BGM" -c:v copy -c:a aac -shortest build/chapters/setsubi_pitch_bgm.mp4
    print('BGM付き → build/chapters/setsubi_pitch_bgm.mp4')
else:
    print('BGM未配置（MyDrive/setsubi_bgm.mp3）。スキップ。')

In [ ]:
# ═══ セル9: 成果物を Drive へ保存 ═══
import shutil, glob, os
OUT = '/content/drive/MyDrive/setsubi_cg_output'
os.makedirs(OUT, exist_ok=True)
for f in glob.glob('build/chapters/setsubi_pitch*.mp4') + glob.glob('build/clips/sp_*.mp4'):
    shutil.copy(f, OUT); print('保存:', os.path.basename(f))
print('→', OUT)